# Plant Disease Classification Using Deep Learning and Transfer Learning

**CENG 476 - Introduction to Deep Learning**  
**Student:** Emir EVREN - **ID:** 210444038

This notebook reflects the **final leakage-audited project state**. The original 99.76% ensemble score is historical only; the final benchmark uses the ultra-strict PlantVillage protocol.

**Final headline:** EfficientNet-B0 **99.01% / 0.9874 Macro-F1**; validation-selected 50/50 ensemble **99.14% / 0.9897**; 3-seed EfficientNet mean **99.001% +/- 0.229 pp**; mapped PlantDoc OOD **23.31% / 25.00%** for EfficientNet / ensemble.


## 1. Setup


In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUTS = PROJECT_ROOT / "outputs"
FULL = OUTPUTS / "audit" / "full_control"
print("Project root:", PROJECT_ROOT)
print("Full-control outputs:", FULL.exists())


## 2. Audit and Final Split

The first image-level protocol was re-audited after its unusually high **99.76%** ensemble accuracy. The audit found **10 exact cross-split duplicate pairs**, **68 strict perceptual near-duplicate pairs**, and **4,956 mapped same-physical-leaf cross-split groups**.

Final ultra-strict split:

| Split | Images |
|---|---:|
| Train | **39,091** |
| Validation | **4,462** |
| Locked official test | **10,709** |
| Total used | **54,262** |

The final protocol quarantined **34 train + 4 validation + 0 test** images from 39 strict `dHash <= 4` cross-split pairs, leaving **0** such pairs. Mapped physical-leaf train/test and train/validation overlap are also **0**.

**Test boundary:** the test set was not used for training, checkpoint selection, hyperparameter tuning, or ensemble-weight selection. Test images were used only in deterministic integrity auditing and later post-hoc diagnostics; test labels/predictions were not used to choose quarantined train/validation images.


## 3. Models and Training

| Model | Parameters | Initialization | Final classifier |
|---|---:|---|---|
| Custom CNN | 399,142 | Random | Dropout(0.40) + Linear(256,38) |
| ResNet18 | 11,196,006 | ImageNet | Dropout(0.30) + Linear(512,38) |
| EfficientNet-B0 | 4,056,226 | ImageNet | Dropout(0.30) + Linear(1280,38) |

Training uses random resized crop, horizontal flip, +/-15 degree rotation, color jitter, ImageNet normalization, AdamW (`weight_decay=1e-4`), `ReduceLROnPlateau`, mixed precision on CUDA, and validation-Macro-F1 checkpoint selection. Transfer models use backbone LR `1e-4` and classifier LR `5e-4`.


## 4. Final Ultra-Strict Results

| Model | Accuracy | Macro-F1 | Weighted-F1 | Macro AUC | Errors |
|---|---:|---:|---:|---:|---:|
| Custom CNN | 84.62% | 0.7813 | 0.8361 | 0.995589 | 1,647 |
| ResNet18 | 97.66% | 0.9686 | 0.9763 | 0.999929 | 251 |
| EfficientNet-B0 | **99.01%** | **0.9874** | **0.9901** | **0.999955** | **106** |
| 50/50 Ensemble | **99.14%** | **0.9897** | **0.9914** | **0.999976** | **92** |

The ensemble weights were selected **only on validation**; 50/50 gave validation Macro-F1 **0.992033**.


In [ ]:
results = pd.DataFrame([
["Custom CNN",84.620413,0.781323,1647],
["ResNet18",97.656177,0.968603,251],
["EfficientNet-B0",99.010178,0.987373,106],
["50/50 Ensemble",99.140910,0.989733,92]],
columns=["Model","Accuracy (%)","Macro-F1","Errors"])
display(results)


## 5. Full-Control Checks

**Generalization gap:** ResNet18 clean-train / validation / test = **98.16 / 98.81 / 97.66%**; EfficientNet-B0 = **99.74 / 98.68 / 99.01%**. This does not support severe conventional train overfitting as the main explanation.

**Calibration / uncertainty:** EfficientNet ECE **0.003845**, 95% bootstrap accuracy CI **98.81-99.20%**; ensemble ECE **0.009121**, CI **98.95-99.31%**.

**Random-label sanity:** chance **2.63%**; true-label validation after shuffled-label training **1.84%**, Macro-F1 **0.0183** -> **PASS**.

**Robustness:** brightness/contrast/JPEG/moderate rotation remain near 98-99%; Gaussian blur drops EfficientNet to **84.08%** and ensemble to **86.53%**; large occlusions cause much larger degradation.


In [ ]:
for name in ["generalization_gap.csv","metrics_calibration_summary.csv","bootstrap_95ci.csv","robustness_stress.csv"]:
    p = FULL / name
    print("\n", name)
    if p.exists(): display(pd.read_csv(p))
    else: print("not found locally")


## 6. Seed Stability

EfficientNet-B0 was retrained on the same fixed ultra-strict manifest. The highest seed is **not** promoted as the final benchmark.

| Seed | Val Macro-F1 | Test accuracy | Test Macro-F1 | Errors |
|---:|---:|---:|---:|---:|
| 42 | 0.982431 | 99.010% | 0.987373 | 106 |
| 123 | 0.986704 | 98.767% | 0.983668 | 132 |
| 777 | 0.992056 | 99.225% | 0.990149 | 83 |

Mean accuracy **99.001%**, standard deviation **0.229 pp**, range **0.458 pp**.


## 7. External OOD Test - PlantDoc

A manually mapped **236-image / 27-source-class** PlantDoc test subset was evaluated without retraining.

| Model | Accuracy | Mapped Macro-F1 |
|---|---:|---:|
| EfficientNet-B0 | **23.31%** | 0.2183 |
| 50/50 Ensemble | **25.00%** | 0.2349 |

PlantDoc and PlantVillage are not identical benchmarks, so this is an **OOD/domain-shift probe**, not a directly comparable replacement test. The large drop demonstrates strong domain dependence and limited real-world generalization.


## 8. Final Interpretation

1. The original image-level split contained leakage risks; **99.76% is not the final benchmark**.
2. Near-99% accuracy remains reproducible after the implemented exact, mapped-leaf, and strict near-duplicate controls.
3. Calibration, repeated seeds, random-label sanity, and same-domain gaps do not strongly support severe conventional overfitting as the main explanation.
4. The PlantDoc drop shows that the model is strongly specialized to the PlantVillage acquisition domain.

> **Final claim:** near-99% PlantVillage performance is reproducible under the audited protocol, but the model is not validated for real-field deployment.


## 9. Reproduction

From the repository root:

```bat
call .\run_ultrastrict_all.bat
call .\run_full_control_all.bat
```

`run_full_control_all.bat` covers calibration, bootstrap CI, per-class/error audit, generalization-gap analysis, corruption/occlusion stress, random-label sanity, Grad-CAM, mapped PlantDoc OOD evaluation, consolidated reporting, and EfficientNet seeds 42/123/777.
